# Day 5 Jetson Orin: Keras to ONNX to TensorRT

This notebook documents the deployment path for the Keras models created in Notebook 09 and Notebook 10. The host computer exports Keras models to ONNX. Jetson Orin Nano builds TensorRT engines from those ONNX files and runs inference.

## Step 1 README: Locate The Course Workspace
This cell finds the repository root and prints the deployment package paths.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent

PACKAGE_DIR = ROOT / 'day 5 jetson orin'
BUNDLE_DIR = PACKAGE_DIR / 'bundle'
ENGINE_DIR = PACKAGE_DIR / 'engines'

print(f'Workspace root: {ROOT}')
print(f'Jetson package: {PACKAGE_DIR}')
print(f'Bundle output: {BUNDLE_DIR}')

## Step 2 README: Check Saved Keras Models
This cell checks whether the Notebook 09 and Notebook 10 Keras models are available. Notebook 10 normally saves its model. If Notebook 09 is missing, the export script can train and save the Fashion-MNIST model once.

In [ ]:
keras_dir = ROOT / 'models' / 'keras'
expected_models = {
    'fashion_mnist_cnn': keras_dir / 'notebook09_fashion_mnist_cnn.keras',
    'cats_dogs_transfer': keras_dir / 'notebook10_cats_dogs_transfer.keras',
}

for name, path in expected_models.items():
    print(f'{name}: {path} -> exists={path.exists()}')

## Step 3 README: Export Keras To ONNX
This cell runs the host export script. It exports both Keras models to ONNX, creates metadata JSON files, and saves sample NumPy inputs for Jetson inference testing.

In [ ]:
import subprocess

export_script = PACKAGE_DIR / 'host' / 'export_models_to_onnx.py'
command = [sys.executable, str(export_script), '--train-missing-fashion']
print('Running:', ' '.join(command))
result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'ONNX export failed with exit code {result.returncode}')

## Step 4 README: Inspect The Bundle
This cell lists the files that must be copied to Jetson Orin Nano. TensorRT engines are not built on the host PC; they are built on the Jetson.

In [ ]:
for path in sorted(BUNDLE_DIR.glob('*')):
    print(path.name, path.stat().st_size, 'bytes')

## Step 5 README: Copy To Jetson Orin Nano
Run this command from a terminal on the host PC. Replace `<JETSON_IP>` with your Jetson address.

In [ ]:
print(f'scp -r "{PACKAGE_DIR}" jetson@<JETSON_IP>:~/edge_ai/')

## Step 6 README: Build TensorRT Engines On Jetson
Run this on Jetson Orin Nano after copying the folder. This converts ONNX files into TensorRT `.engine` files.

In [ ]:
print('cd ~/edge_ai/day\\ 5\\ jetson\\ orin')
print('chmod +x jetson/build_tensorrt_engines.sh')
print('./jetson/build_tensorrt_engines.sh --precision fp16')

## Step 7 README: Run Jetson Inference
Run these commands on Jetson to test each TensorRT engine with the bundled sample inputs.

In [ ]:
print('python3 jetson/infer_jetson_orin.py --model fashion --precision fp16')
print('python3 jetson/infer_jetson_orin.py --model catsdogs --precision fp16')
print('python3 jetson/infer_jetson_orin.py --model catsdogs --image /path/to/image.jpg --precision fp16')